In [1]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

BASE = '/Users/abilindemann/projects/sa-housing-food-insecurity/data/processed/'

food = pd.read_csv(BASE + 'food_scores.csv', dtype={'GEOID': str})

print(f"Loaded {len(food)} block groups")
print(f"High housing insecurity: {food['housing_insecurity_high'].sum()}")
print(f"High food insecurity:    {food['food_insecurity_high'].sum()}")
print(f"Nulls in housing_score: {food['housing_score'].isnull().sum()}")

Loaded 1139 block groups
High housing insecurity: 277
High food insecurity:    616
Nulls in housing_score: 31


In [3]:
# Quadrant classification — cross housing and food insecurity flags
# Both flags are already binary (1 = high, 0 = low) from previous notebooks

conditions = [
    (food['housing_insecurity_high'] == 1) & (food['food_insecurity_high'] == 1),
    (food['housing_insecurity_high'] == 1) & (food['food_insecurity_high'] == 0),
    (food['housing_insecurity_high'] == 0) & (food['food_insecurity_high'] == 1),
    (food['housing_insecurity_high'] == 0) & (food['food_insecurity_high'] == 0),
]
labels = [
    'High Housing + High Food',
    'High Housing Only',
    'High Food Only',
    'Low Both',
]

food['quadrant'] = np.select(conditions, labels, default='Unknown')

print("Quadrant distribution:")
print(food['quadrant'].value_counts())
print(f"\nTotal classified: {(food['quadrant'] != 'Unknown').sum()}")
print(f"Unclassified (missing data): {(food['quadrant'] == 'Unknown').sum()}")

Quadrant distribution:
quadrant
High Food Only              461
Low Both                    401
High Housing + High Food    155
High Housing Only           122
Name: count, dtype: int64

Total classified: 1139
Unclassified (missing data): 0


In [5]:
def latlon_to_xy(lat, lon):
    x = lon * 59.0
    y = lat * 69.0
    return x, y

# Load service providers
food_banks = pd.read_csv(BASE + 'food_banks_bexar.csv')
wic = pd.read_csv(BASE + 'wic_clinics_bexar.csv')

print(f"Food banks/pantries: {len(food_banks)}")
print(f"WIC clinics: {len(wic)}")
print(f"\nFood bank columns: {food_banks.columns.tolist()}")
print(f"WIC columns: {wic.columns.tolist()}")

Food banks/pantries: 10
WIC clinics: 15

Food bank columns: ['name', 'address', 'city', 'latitude', 'longitude']
WIC columns: ['name', 'address', 'city', 'latitude', 'longitude']


In [7]:
# Build KD-trees for food banks and WIC clinics
fb_x, fb_y = latlon_to_xy(food_banks['latitude'].values, food_banks['longitude'].values)
wic_x, wic_y = latlon_to_xy(wic['latitude'].values, wic['longitude'].values)

fb_tree  = cKDTree(np.column_stack([fb_x, fb_y]))
wic_tree = cKDTree(np.column_stack([wic_x, wic_y]))

# Query nearest service for each block group centroid
bg_x, bg_y = latlon_to_xy(food['lat'].values, food['lon'].values)
coords = np.column_stack([bg_x, bg_y])

fb_dist, fb_idx   = fb_tree.query(coords, k=1)
wic_dist, wic_idx = wic_tree.query(coords, k=1)

food['nearest_food_bank_miles'] = fb_dist
food['nearest_food_bank_name']  = food_banks['name'].values[fb_idx]
food['nearest_wic_miles']       = wic_dist
food['nearest_wic_name']        = wic['name'].values[wic_idx]

print("Distance to nearest food bank (miles):")
print(food['nearest_food_bank_miles'].describe().round(3))
print("\nDistance to nearest WIC clinic (miles):")
print(food['nearest_wic_miles'].describe().round(3))

Distance to nearest food bank (miles):
count    1139.000
mean        4.023
std         2.993
min         0.073
25%         1.905
50%         3.393
75%         5.121
max        16.696
Name: nearest_food_bank_miles, dtype: float64

Distance to nearest WIC clinic (miles):
count    1139.000
mean        3.322
std         2.747
min         0.078
25%         1.431
50%         2.510
75%         4.272
max        15.185
Name: nearest_wic_miles, dtype: float64


In [9]:
# Flag block groups with high insecurity AND poor service access
# Thresholds: >3 miles to food bank, >3 miles to WIC (above median for both)

food['service_gap'] = (
    (food['quadrant'] == 'High Housing + High Food') &
    (food['nearest_food_bank_miles'] > 3.0) &
    (food['nearest_wic_miles'] > 3.0)
).astype(int)

print(f"Dual insecurity block groups:                    {(food['quadrant'] == 'High Housing + High Food').sum()}")
print(f"Dual insecurity + poor service access:           {food['service_gap'].sum()}")
print(f"\nAmong dual insecurity block groups:")
dual = food[food['quadrant'] == 'High Housing + High Food']
print(f"  Mean distance to food bank: {dual['nearest_food_bank_miles'].mean():.2f} miles")
print(f"  Mean distance to WIC:       {dual['nearest_wic_miles'].mean():.2f} miles")
print(f"  >3 miles to food bank:      {(dual['nearest_food_bank_miles'] > 3.0).sum()}")
print(f"  >3 miles to WIC:            {(dual['nearest_wic_miles'] > 3.0).sum()}")

Dual insecurity block groups:                    155
Dual insecurity + poor service access:           72

Among dual insecurity block groups:
  Mean distance to food bank: 4.69 miles
  Mean distance to WIC:       3.79 miles
  >3 miles to food bank:      99
  >3 miles to WIC:            73


In [ ]:
out_cols = [
    'GEOID', 'lat', 'lon',
    'housing_score', 'housing_insecurity_high',
    'nearest_grocery_miles', 'food_distance_norm',
    'within_half_mile', 'within_one_mile',
    'LILATracts_1And10', 'LILATracts_halfAnd10',
    'food_insecurity_high',
    'quadrant', 'service_gap',
    'nearest_food_bank_miles', 'nearest_food_bank_name',
    'nearest_wic_miles', 'nearest_wic_name',
    'nearest_grocery_name',
    'B19013_001E', 'poverty_rate', 'pct_hispanic',
]

out = food[[c for c in out_cols if c in food.columns]].copy()
out.to_csv(BASE + 'overlap_scores.csv', index=False)
print(f"Saved overlap_scores.csv with {len(out)} rows and {len(out.columns)} columns")

## Notebook 03 — Overlap Analysis

**Output:** `data/processed/overlap_scores.csv`

**Quadrant classification** (housing insecurity × food insecurity):
- High Housing + High Food: 155 block groups (13.6%) — dual insecurity
- High Food Only:           461 block groups (40.5%)
- High Housing Only:        122 block groups (10.7%)
- Low Both:                 401 block groups (35.2%)

**Service gap findings:**
- Average distance to nearest food bank: 4.0 miles countywide; 4.7 miles in dual-insecurity areas
- Average distance to nearest WIC clinic: 3.3 miles countywide; 3.8 miles in dual-insecurity areas
- 72 block groups have high dual insecurity AND are >3 miles from both a food bank and WIC clinic